# ⚖️ LawGPT-IN — Stage 3: Hybrid RAG Backend

```
User Question
     │
     ├─── BM25 (keyword)   ──┐
     │                       ├──► Reciprocal Rank Fusion ──► Top-5 Chunks
     └─── FAISS (semantic) ──┘                                    │
                                                                   ▼
                                               LawGPT-IN (LoRA) + Context
                                                                   │
                                                             Final Answer
```

| Cell | Stage | Saves to disk? |
|------|-------|----------------|
| 1 | Install packages | — |
| 2 | Config & Secrets | — |
| 3 | Load LoRA model | — |
| 4 | Build FAISS index | `lawgpt_faiss.index` + `lawgpt_chunks.json` |
| 5 | Build BM25 index | in-memory |
| 6 | Test hybrid retrieval | — |
| 7 | Define FastAPI app | — |
| 8 | Launch server + ngrok | — |
| 9 | Test live API | — |

> **Resume tip:** If Colab crashes after Cell 4, re-run Cells 1→2→3→4 (skips rebuild) → 5 → 7 → 8

---
## 📦 Cell 1 — Install Packages
Run once per session.

In [ ]:
# ── CELL 1 : Install packages ─────────────────────────────────
try:
    import unsloth
    import pyngrok
    import faiss
    print('✅  Packages already installed. Skipping installation.')
except ImportError:
    print('⏳  Installing packages... (this may take a minute)')
    # Using -q (quiet) instead of %%capture so our print statements still show
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
    !pip install -q faiss-gpu sentence-transformers rank_bm25
    !pip install -q fastapi uvicorn nest-asyncio pyngrok
    !pip install -q datasets huggingface_hub
    print('✅  All packages installed.')

!pip install -q faiss-gpu-cu12 sentence-transformers rank_bm25 fastapi uvicorn nest-asyncio pyngrok datasets huggingface_hub





---
## ⚙️ Cell 2 — Config & Secrets

**Before running:**
1. Click the 🔑 key icon in the Colab left sidebar
2. Add `HF_TOKEN` → your HuggingFace write token
3. Add `NGROK_TOKEN` → free at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# ── CELL 2 : Config & Secrets ─────────────────────────────────
# MUST re-run every time Colab restarts.

from google.colab import userdata
import torch
import os

# ── Tokens from Colab Secrets (never hardcode!) ───────────────
HF_TOKEN    = userdata.get('HF_TOKEN')
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

# ── Model & dataset IDs ───────────────────────────────────────
HF_DATASET_ID = "SCARA02/repo_lawgpt"
MODEL_ID      = "SCARA02/lawgpt-mistral-7b-v1"
BASE_MODEL_ID = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"

# ── Retrieval config ──────────────────────────────────────────
TOP_K_BM25    = 10    # BM25 candidate pool
TOP_K_FAISS   = 10    # FAISS candidate pool
TOP_K_FINAL   = 5     # chunks sent to LLM after RRF fusion
CHUNK_SIZE    = 400   # characters per chunk
CHUNK_OVERLAP = 80    # overlap between consecutive chunks
EMBED_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"

# ── System prompt ─────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are LawGPT-IN, an expert AI assistant specialising in Indian law. "
    "You have deep knowledge of IPC, CrPC, Constitution of India, and Indian court judgements. "
    "Answer using ONLY the provided legal context. "
    "Always cite the specific acts and sections mentioned in the context. "
    "If the context does not contain enough information, say so clearly."
)

print('✅  Config loaded.')
print(f'   Dataset   → {HF_DATASET_ID}')
print(f'   Model     → {MODEL_ID}')
print(f'   GPU       → {torch.cuda.get_device_name(0)}')
print(f'   VRAM      → {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

---
## 🤖 Cell 3 — Load Fine-tuned LoRA Model

Loads Mistral-7B in 4-bit then applies your LawGPT LoRA adapters on top.

In [ ]:
import os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# Monkey-patch the statistics check to do nothing
import unsloth.models._utils as _unsloth_utils
_unsloth_utils.get_statistics = lambda *args, **kwargs: None

from peft import PeftModel
from huggingface_hub import login

login(token=HF_TOKEN)

print('📥  Loading base model (4-bit)...')
# Use ModelScope as a temporary measure if HuggingFace is down
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL_ID,
    max_seq_length = 2048,
    dtype          = None,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5 min timeout instead of default

tokenizer = get_chat_template(tokenizer, chat_template="chatml")
FastLanguageModel.for_inference(model)

print('✅  Model ready.')
print(f'   VRAM used : {torch.cuda.memory_allocated()/1e9:.2f} GB')

---
## 🗂️ Cell 4 — Build FAISS Index

**Resumable:** If `lawgpt_faiss.index` already exists on disk, this cell skips the build and loads directly.  

Pipeline: Dataset → chunk text → embed with MiniLM → store in FAISS IndexFlatIP (cosine similarity).

In [ ]:
# ── CELL 4 : Build FAISS index ────────────────────────────────
# Resumable: skips build if index already saved to disk.

import json, re, numpy as np, faiss
from pathlib import Path
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

INDEX_PATH  = Path("lawgpt_faiss.index")
CHUNKS_PATH = Path("lawgpt_chunks.json")

# ── Always load the embedder ───────────────────────────────────
print('📥  Loading embedding model...')
embedder = SentenceTransformer(EMBED_MODEL, device="cuda")
print('✅  Embedder ready.')

# ── Resume: reload if already built ───────────────────────────
if INDEX_PATH.exists() and CHUNKS_PATH.exists():
    print('\n⏭️   FAISS index found on disk — loading...')
    faiss_index = faiss.read_index(str(INDEX_PATH))
    chunks      = json.loads(CHUNKS_PATH.read_text())
    print(f'✅  Loaded {faiss_index.ntotal} vectors | {len(chunks)} chunks.')

else:
    # ── Helper functions ──────────────────────────────────────
    def extract_user_text(chatml):
        m = re.search(r'<\|im_start\|>user\n(.*?)<\|im_end\|>', chatml, re.DOTALL)
        return m.group(1).strip() if m else ""

    def extract_assistant_text(chatml):
        m = re.search(r'<\|im_start\|>assistant\n(.*?)<\|im_end\|>', chatml, re.DOTALL)
        return m.group(1).strip() if m else ""

    def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
        result, start = [], 0
        while start < len(text):
            result.append(text[start:start+size])
            start += size - overlap
        return result

    # ── Load dataset ──────────────────────────────────────────
    print('\n📥  Loading dataset...')
    ds = load_dataset(HF_DATASET_ID, split="train", token=HF_TOKEN)
    print(f'   {len(ds)} rows.')

    # ── Chunk ─────────────────────────────────────────────────
    print('✂️   Chunking...')
    chunks = []
    for row in tqdm(ds, desc='Chunking'):
        full_text = f"Q: {extract_user_text(row['text'])}\nA: {extract_assistant_text(row['text'])}"
        for chunk in chunk_text(full_text):
            if len(chunk.strip()) > 80:
                chunks.append({
                    "text":  chunk,
                    "court": row.get("court", ""),
                    "type":  row.get("type",  ""),
                    "tid":   row.get("tid",   ""),
                })
    print(f'   Total chunks: {len(chunks)}')

    # ── Embed ─────────────────────────────────────────────────
    print('🔢  Embedding (batch=128)...')
    embeddings = embedder.encode(
        [c["text"] for c in chunks],
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)

    # ── Build & save FAISS ────────────────────────────────────
    print('\n🏗️   Building FAISS index...')
    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    gpu_index   = faiss.index_cpu_to_gpu(faiss.StandardGpuResources(), 0, faiss_index)
    gpu_index.add(embeddings)
    faiss_index = faiss.index_gpu_to_cpu(gpu_index)

    faiss.write_index(faiss_index, str(INDEX_PATH))
    CHUNKS_PATH.write_text(json.dumps(chunks, ensure_ascii=False))

    print(f'✅  FAISS index saved: {faiss_index.ntotal} vectors.')

---
## 🔤 Cell 5 — Build BM25 Index

BM25 catches **exact legal terms** like `Section 498A` or `Negotiable Instruments Act` that semantic search often misses.  
Both BM25 + FAISS results are fused via **Reciprocal Rank Fusion (RRF)**.

In [ ]:
# ── CELL 5 : Build BM25 index ─────────────────────────────────
# In-memory — takes ~30s. Must re-run if Colab restarts.

from rank_bm25 import BM25Okapi
import re

LEGAL_KEEP = {
    'section','act','crpc','ipc','court','bail','appeal',
    'petition','tribunal','order','held','judgment',
    'judgement','verdict','conviction','writ','habeas'
}
STOPWORDS = {
    'the','a','an','is','in','of','to','and','or','for',
    'be','by','at','it','on','as','was','are','has','have'
}

def legal_tokenize(text):
    text   = re.sub(r'([A-Za-z]+)(\d+)', r'\1 \2', text)
    tokens = re.findall(r'[a-zA-Z0-9]+', text.lower())
    return [t for t in tokens if t not in STOPWORDS or t in LEGAL_KEEP]

print('🔍  Building BM25 index...')
corpus_tokens = [legal_tokenize(c["text"]) for c in tqdm(chunks, desc='Tokenising')]
bm25          = BM25Okapi(corpus_tokens)

print(f'✅  BM25 index ready: {len(chunks)} chunks.')

---
## 🧪 Cell 6 — Test Hybrid Retrieval

Test BM25 + FAISS + RRF fusion **without** calling the LLM. Fast to iterate on.

In [ ]:
# ── CELL 6 : Test hybrid retrieval ────────────────────────────

import numpy as np

def reciprocal_rank_fusion(bm25_ids, faiss_ids, k=60):
    """
    RRF score = Σ 1/(k + rank)
    Merges two ranked lists. k=60 is the standard constant.
    """
    scores = {}
    for rank, idx in enumerate(bm25_ids):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)
    for rank, idx in enumerate(faiss_ids):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)
    return sorted(scores, key=lambda x: scores[x], reverse=True)


def hybrid_retrieve(query, top_k_final=TOP_K_FINAL):
    # BM25 — keyword
    bm25_scores = bm25.get_scores(legal_tokenize(query))
    bm25_ids    = np.argsort(bm25_scores)[::-1][:TOP_K_BM25].tolist()

    # FAISS — semantic
    q_emb = embedder.encode(
        [query], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)
    _, faiss_raw = faiss_index.search(q_emb, TOP_K_FAISS)
    faiss_ids    = faiss_raw[0].tolist()

    # RRF fusion
    fused = reciprocal_rank_fusion(bm25_ids, faiss_ids)
    return [chunks[i] for i in fused[:top_k_final] if i < len(chunks)]


# ── Test ──────────────────────────────────────────────────────
test_query = "What is anticipatory bail under Section 438 CrPC?"
retrieved  = hybrid_retrieve(test_query)

print(f'Query: "{test_query}"')
print(f'\nTop {len(retrieved)} retrieved chunks:\n')
for i, c in enumerate(retrieved):
    print(f'  [{i+1}] court={c["court"]} | type={c["type"]}')
    print(f'       {c["text"][:180]}...')
    print()

---
## 🌐 Cell 7 — Define FastAPI App

Registers all endpoints. Does **not** start the server yet — that's Cell 8.

| Endpoint | Method | Purpose |
|---|---|---|
| `/` | GET | Status check |
| `/health` | GET | Model + index stats |
| `/query` | POST | Full RAG answer |
| `/retrieve` | GET | Debug retrieval only (no LLM) |

In [ ]:
# ── CELL 7 : FastAPI app ───────────────────────────────────────

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import time

app = FastAPI(
    title       = "LawGPT-IN API",
    description = "Hybrid RAG over Indian court judgements",
    version     = "1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_credentials=True,
    allow_methods=["*"], allow_headers=["*"],
)


# ── Schemas ───────────────────────────────────────────────────
class QueryRequest(BaseModel):
    question:   str
    top_k:      Optional[int] = 5
    max_tokens: Optional[int] = 512

class SourceChunk(BaseModel):
    text:  str
    court: str
    type:  str

class QueryResponse(BaseModel):
    answer:     str
    sources:    list[SourceChunk]
    latency_ms: float


# ── RAG core ──────────────────────────────────────────────────
def build_rag_prompt(question, retrieved_chunks):
    context = "\n\n".join(
        f"[Source {i+1} | {c['court']} | {c['type']}]\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    )
    return (
        f"Use the following Indian court judgement excerpts to answer the question.\n\n"
        f"--- LEGAL CONTEXT ---\n{context}\n--- END CONTEXT ---\n\n"
        f"Question: {question}"
    )


def rag_generate(question, top_k=5, max_new_tokens=512):
    t0        = time.time()
    retrieved = hybrid_retrieve(question, top_k_final=top_k)
    messages  = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_rag_prompt(question, retrieved)},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids          = inputs,
            max_new_tokens     = max_new_tokens,
            use_cache          = True,
            temperature        = 0.3,
            do_sample          = True,
            repetition_penalty = 1.1,
        )

    raw = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
    if "<|im_start|>assistant\n" in raw:
        answer = raw.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    else:
        answer = raw.strip()

    return answer, retrieved, round((time.time() - t0) * 1000, 1)


# ── Endpoints ─────────────────────────────────────────────────
@app.get("/")
def root():
    return {"status": "LawGPT-IN RAG API is running ⚖️"}


@app.get("/health")
def health():
    return {
        "status":     "ok",
        "model":      MODEL_ID,
        "chunks":     len(chunks),
        "vectors":    faiss_index.ntotal,
        "gpu_mem_gb": round(torch.cuda.memory_allocated() / 1e9, 2),
    }


@app.post("/query", response_model=QueryResponse)
def query(req: QueryRequest):
    if not req.question.strip():
        raise HTTPException(400, "Question cannot be empty.")
    if len(req.question) > 2000:
        raise HTTPException(400, "Question too long (max 2000 chars).")

    answer, retrieved, latency = rag_generate(
        req.question, top_k=req.top_k, max_new_tokens=req.max_tokens
    )
    sources = [
        SourceChunk(text=c["text"][:300], court=c["court"], type=c["type"])
        for c in retrieved
    ]
    return QueryResponse(answer=answer, sources=sources, latency_ms=latency)


@app.get("/retrieve")
def retrieve_only(q: str, top_k: int = 5):
    """Debug: see retrieved chunks without calling the LLM."""
    return {"query": q, "chunks": hybrid_retrieve(q, top_k_final=top_k)}


print('✅  FastAPI app defined.')
print('   GET  /           → status')
print('   GET  /health     → model + index stats')
print('   POST /query      → full RAG answer')
print('   GET  /retrieve   → debug retrieval')
print('\n→  Run Cell 8 to start the server.')

---
## 🚀 Cell 8 — Launch Server + ngrok Tunnel

Starts the FastAPI server and exposes it publicly via ngrok.  
**After running this cell:**
1. Copy the `Public URL` printed below
2. Paste it into your Next.js `.env.local` as `NEXT_PUBLIC_API_URL=<url>`

> ⚠️ Keep this cell running. If it stops, re-run this cell only.

In [ ]:
# ── CELL 8 : Launch server + ngrok ────────────────────────────

import nest_asyncio, uvicorn, threading, time
from pyngrok import ngrok, conf

nest_asyncio.apply()

# Setup ngrok
conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()   # clear any leftover tunnels

PORT = 8000

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)  # wait for server to be ready

# Open public tunnel
tunnel     = ngrok.connect(PORT, "http")
public_url = tunnel.public_url

print('🚀  LawGPT-IN RAG API is LIVE!')
print('=' * 55)
print(f'   Public URL  :  {public_url}')
print(f'   Health      :  {public_url}/health')
print(f'   API Docs    :  {public_url}/docs')
print('=' * 55)
print('\n📋  Add this to your Next.js .env.local:')
print(f'\n   NEXT_PUBLIC_API_URL={public_url}\n')
print('⚠️  Keep this cell running!')

---
## ✅ Cell 9 — Test the Live API

Verifies all endpoints are working end-to-end.

In [ ]:
# ── CELL 9 : Test live API ─────────────────────────────────────

import requests as req

API = public_url   # set from Cell 8

# ── Test 1: Root ──────────────────────────────────────────────
print('1️⃣   Root check...')
r = req.get(f"{API}/")
print(f'   {r.json()}\n')

# ── Test 2: Health ────────────────────────────────────────────
print('2️⃣   Health check...')
r = req.get(f"{API}/health")
h = r.json()
print(f'   Model   : {h["model"]}')
print(f'   Chunks  : {h["chunks"]}')
print(f'   Vectors : {h["vectors"]}')
print(f'   GPU mem : {h["gpu_mem_gb"]} GB\n')

# ── Test 3: Retrieve only (fast, no LLM) ──────────────────────
print('3️⃣   Retrieval test (no LLM)...')
r   = req.get(f"{API}/retrieve", params={"q": "Section 438 bail CrPC", "top_k": 3})
ret = r.json()
for i, c in enumerate(ret["chunks"]):
    print(f'   [{i+1}] {c["court"]} | {c["text"][:100]}...')

# ── Test 4: Full RAG query ────────────────────────────────────
print('\n4️⃣   Full RAG query (takes 20–40s on T4)...')
r = req.post(
    f"{API}/query",
    json={"question": "What is anticipatory bail under CrPC?", "top_k": 5, "max_tokens": 300},
    timeout=120,
)
resp = r.json()

print(f'\n⏱️   Latency  : {resp["latency_ms"]} ms')
print(f'\n📜  Answer:\n{resp["answer"]}')
print(f'\n📚  Sources used: {len(resp["sources"])}')
for i, s in enumerate(resp["sources"]):
    print(f'   [{i+1}] {s["court"]} — {s["text"][:80]}...')

---
## ✅ All Done!

**Resume guide — if Colab crashes:**

| Crashed at | Re-run these cells |
|---|---|
| Cell 4 (FAISS build) | 1 → 2 → 3 → 4 (resumes from disk) → 5 → 7 → 8 |
| Cell 5 (BM25) | 2 → 5 → 7 → 8 |
| Cell 8 (server) | 2 → 6 → 7 → 8 |

**Next step:** Set up the Next.js frontend in your local machine.
```bash
npx create-next-app@latest lawgpt-ui
# paste your ngrok URL into .env.local
```